## SAM 3 ENTRENADO

In [1]:
from scipy.ndimage import binary_erosion
from scipy.spatial import cKDTree
from scipy.spatial.distance import directed_hausdorff
import numpy as np

def get_boundary(mask, width=2):
    eroded = binary_erosion(mask, iterations=width)
    return mask & ~eroded

def boundary_iou(pred_mask, gt_mask, width=2):
    pred_boundary = get_boundary(pred_mask, width)
    gt_boundary   = get_boundary(gt_mask, width)
    intersection  = np.logical_and(pred_boundary, gt_boundary).sum()
    union         = np.logical_or(pred_boundary, gt_boundary).sum()
    return intersection / union if union > 0 else 0

# TARDA MUCHO PORQUE TIENE QUE COGER TODOS LOS PUNTOS DE LA MÁSCARA
"""def assd(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    
    d_pred_to_gt = cKDTree(gt_points).query(pred_points)[0].mean()
    d_gt_to_pred = cKDTree(pred_points).query(gt_points)[0].mean()
    
    return (d_pred_to_gt + d_gt_to_pred) / 2"""

def hausdorff_95(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    d1 = directed_hausdorff(pred_points, gt_points)[0]
    d2 = directed_hausdorff(gt_points, pred_points)[0]
    return np.percentile([d1, d2], 95)


In [2]:
import time
import torch

def measure_inference(predictor, image, point_coords, point_labels):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    
    start   = time.time()
    predictor.set_image(image)
    masks, scores, _ = predictor.predict(point_coords=point_coords, point_labels=point_labels)
    latency = (time.time() - start) * 1000  # Está en ms

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return masks, scores, latency, vram

def measure_inference_sam3_points(model, img_path, central_point):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    start = time.time()
    results = model(img_path, points=central_point, labels=[1], verbose=False)
    latency = (time.time() - start) * 1000
    vram = torch.cuda.memory_allocated() / 1024**2 - vram_before if torch.cuda.is_available() else 0
    return results, latency, vram

def measure_inference_sam3_prompt(predictor, img_path, text_prompt):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2

    start = time.time()
    predictor.set_image(img_path)
    results = predictor(text=[text_prompt])
    latency = (time.time() - start) * 1000

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return results, latency, vram

In [3]:
import numpy as np

def compute_all_metrics(pred_mask, gt_mask):
    """a"""
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()

    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f2 = 5 * (precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0
    pixel_accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0
    return iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy

In [4]:
import os
import shutil
import random

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"

def split_dataset(dataset, train_ratio=0.7, val_ratio=0.15):
    images_dir = os.path.join(dataset, "images")
    masks_dir = os.path.join(dataset, "masks")

    images = [f.replace(".jpg", "") for f in os.listdir(images_dir) if f.endswith(".jpg")]
    random.seed(42)
    random.shuffle(images)

    n = len(images)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    splits = {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train+n_val:]
    }

    for split, names in splits.items():
        os.makedirs(os.path.join(dataset, "images", split), exist_ok=True)
        os.makedirs(os.path.join(dataset, "masks",  split), exist_ok=True)
        for name in names:
            shutil.copy(os.path.join(images_dir, name + ".jpg"), os.path.join(dataset, "images", split, name + ".jpg"))
            shutil.copy(os.path.join(masks_dir,  name + ".jpg"), os.path.join(dataset, "masks",  split, name + ".jpg"))

split_dataset(dataset)


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from ultralytics import SAM
from ultralytics.models.sam import SAM3Predictor
import cv2
import numpy as np
import os
import json

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, bbox_json, img_size=1008):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir  = os.path.join(dataset_path, "masks",  split)
        self.samples    = []

        with open(bbox_json) as f:
            bbox_json = json.load(f)

        for img_name, info in bbox_json.items():
            img_path  = os.path.join(self.images_dir, img_name + ".jpg")
            mask_path = os.path.join(self.masks_dir,  img_name + ".jpg")
            if os.path.exists(img_path) and os.path.exists(mask_path):
                self.samples.append((img_path, mask_path, info["bbox"][0]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, bbox = self.samples[idx]

        image = cv2.imread(img_path)
        orig_h, orig_w = image.shape[:2]
        scale_x = self.img_size / orig_w
        scale_y = self.img_size / orig_h
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (288, 288))
        mask = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        xmin, ymin, xmax, ymax = bbox["xmin"], bbox["ymin"], bbox["xmax"], bbox["ymax"]
        point = torch.tensor([[(xmin + (xmax-xmin)/2) * scale_x, (ymin + (ymax-ymin)/2) * scale_y]]).float()
        label = torch.tensor([1])

        return image, mask, point, label


def train_sam(dataset_path, bbox_json, weights_path, output_weights, epochs=50, batch_size=4):
    sam3_wrapper = SAM(weights_path)
    sam3 = sam3_wrapper.model
    for param in sam3.parameters():
        param.data = param.data.to(device)
    for buffer in sam3.buffers():
        buffer.data = buffer.data.to(device)
    sam3.to(device)

    for param in sam3.image_encoder.parameters():
        param.requires_grad = False
    for param in sam3.sam_prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam3.sam_mask_decoder.parameters(), lr=1e-4)
    loss_fn   = nn.BCEWithLogitsLoss()

    train_dataset = SegDataset(dataset_path, "train", bbox_json)
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    predictor = SAM3Predictor(overrides={"model": weights_path, "task": "segment", "mode": "predict", "verbose": False})
    predictor.setup_model(sam3)

    predictor._bb_feat_sizes = [(288, 288), (144, 144), (72, 72)]
    
    sam3.train()

    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            masks  = masks.to(device)
            points = points.to(device)
            labels = labels.to(device)

            loss_batch = 0
            for i in range(images.shape[0]):

                with torch.no_grad():
                    image_tensor = images[i].unsqueeze(0).to(device)
                    backbone_out = sam3.forward_image(image_tensor)
                    _, vision_feats, _, _ = sam3._prepare_backbone_features(backbone_out)
                    #print(len(vision_feats), [v.shape for v in vision_feats])
                    if sam3.directly_add_no_mem_embed:
                        vision_feats[-1] = vision_feats[-1] + sam3.no_mem_embed
                    feats = [
                        feat.permute(1, 2, 0).reshape(1, -1, *feat_size)
                        for feat, feat_size in zip(vision_feats, predictor._bb_feat_sizes)
                    ]
                    features = {"image_embed": feats[-1], "high_res_feats": feats[:-1]}

                sparse_embeddings, dense_embeddings = sam3.sam_prompt_encoder(
                    points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                    boxes=None,
                    masks=None
                )

                image_embedding = features["image_embed"]
                image_pe        = sam3.sam_prompt_encoder.get_dense_pe()
                high_res_features = features["high_res_feats"]

                low_res_masks, _, _, _ = sam3.sam_mask_decoder(
                    image_embeddings=image_embedding,
                    image_pe=image_pe,
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                    repeat_image=False,
                    high_res_features=high_res_features,
                )

                loss_batch += loss_fn(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    torch.save({"model": sam3.state_dict()}, output_weights)
    return output_weights


## SAM 3 (Vía Puntos) 

In [6]:
import numpy as np
import os
import cv2
import pandas as pd
import json
import torch

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"
bboxes = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG\\kavsir_bboxes.json"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam3_kvasir.pt"


def evaluate_model(dataset, weights_path):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    sam3_wrapper = SAM(weights)
    sam3 = sam3_wrapper.model
    for param in sam3.parameters():
        param.data = param.data.to(device)
    for buffer in sam3.buffers():
        buffer.data = buffer.data.to(device)
    sd = torch.load(weights_path)["model"]
    sam3.load_state_dict(sd)
    sam3.eval()

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", "test", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  "test", img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_sam3_points(sam3_wrapper, img_path, np.array(central_point))

        if results is None or results[0].masks is None:
            continue
        scores = results[0].boxes.conf.cpu().numpy()
        best_idx  = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    resultados = {
         "modelo": ["sam_3_kvasir"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
         df.to_csv(output_path, index=False)

    print(f"Mean IoU: {mean_iou:.4f}")
    print(f"Mean F1 Score: {mean_f1:.4f}")
    print(f"Mean Recall: {mean_recall:.4f}")
    print(f"Mean Precision: {mean_precision:.4f}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Mean Specificity: {mean_specificity:.4f}")
    print(f"Mean F2: {mean_f2:.4f}")
    print(f"Mean Pixel Accuracy: {mean_pixel_accuracy:.4f}")
    print(f"Mean Boundary IoU: {mean_boundary_iou:.4f}")
    print(f"Mean Hausdorff-95: {mean_hausdorff_95:.4f}")
    print(f"Mean Latency (ms): {mean_latency:.2f}")
    print(f"Mean VRAM Usage (MB): {mean_vram:.2f}")
  
trained_weights = train_sam(dataset, bboxes, weights, output_weights)
evaluate_model(dataset, trained_weights)


c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ultralytics 8.4.26  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)


C:\Users\DanielTalavera\AppData\Local\Temp\ipykernel_896\3491141125.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(weights_path)["model"]


WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must

## SAM 3 (Vía Prompts)

In [8]:
import numpy as np
import os
import cv2
import pandas as pd
import json
import torch
from ultralytics.models.sam import SAM3SemanticPredictor

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"
bboxes = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG\\kavsir_bboxes.json"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam3_prompts_kvasir.pt"


def evaluate_model(dataset, weights_path):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)
    
    overrides = dict(conf=0.25, task="segment", mode="predict", model=weights, verbose=False)
    predictor = SAM3SemanticPredictor(overrides=overrides)
    predictor.setup_model(model=None)
    print([name for name, _ in predictor.model.named_children()])
    sd = torch.load(weights_path, map_location=device)
    model_sd = sd["model"] if "model" in sd else sd
    predictor.model.load_state_dict(model_sd)
    predictor.model.eval()
    predictor.model.to(device)

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", "test", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  "test", img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)    

        results, latency, vram = measure_inference_sam3_prompt(predictor, img_path, "polyp")

        if results is None or results[0].masks is None:
            print(f"Sin máscara: {img_name}")
            continue
        pred_mask = results[0].masks.data[0].cpu().numpy().astype(bool)
        if len(results[0].masks.data) > 1:
            areas = [m.sum().item() for m in results[0].masks.data]
            pred_mask = results[0].masks.data[np.argmax(areas)].cpu().numpy().astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    resultados = {
         "modelo": ["sam_3_prompts_kvasir"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
         df.to_csv(output_path, index=False)

    print(f"Mean IoU: {mean_iou:.4f}")
    print(f"Mean F1 Score: {mean_f1:.4f}")
    print(f"Mean Recall: {mean_recall:.4f}")
    print(f"Mean Precision: {mean_precision:.4f}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Mean Specificity: {mean_specificity:.4f}")
    print(f"Mean F2: {mean_f2:.4f}")
    print(f"Mean Pixel Accuracy: {mean_pixel_accuracy:.4f}")
    print(f"Mean Boundary IoU: {mean_boundary_iou:.4f}")
    print(f"Mean Hausdorff-95: {mean_hausdorff_95:.4f}")
    print(f"Mean Latency (ms): {mean_latency:.2f}")
    print(f"Mean VRAM Usage (MB): {mean_vram:.2f}")
  
trained_weights = train_sam(dataset, bboxes, weights, output_weights)
evaluate_model(dataset, trained_weights)


Ultralytics 8.4.26  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
Ultralytics 8.4.26  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
['backbone', 'geometry_encoder', 'transformer', 'segmentation_head', 'dot_prod_scoring']


C:\Users\DanielTalavera\AppData\Local\Temp\ipykernel_1628\1805649311.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(weights_path, map_location=device)


RuntimeError: Error(s) in loading state_dict for SAM3SemanticModel:
	Missing key(s) in state_dict: "backbone.vision_backbone.trunk.pos_embed", "backbone.vision_backbone.trunk.patch_embed.proj.weight", "backbone.vision_backbone.trunk.blocks.0.norm1.weight", "backbone.vision_backbone.trunk.blocks.0.norm1.bias", "backbone.vision_backbone.trunk.blocks.0.attn.qkv.weight", "backbone.vision_backbone.trunk.blocks.0.attn.qkv.bias", "backbone.vision_backbone.trunk.blocks.0.attn.proj.weight", "backbone.vision_backbone.trunk.blocks.0.attn.proj.bias", "backbone.vision_backbone.trunk.blocks.0.norm2.weight", "backbone.vision_backbone.trunk.blocks.0.norm2.bias", "backbone.vision_backbone.trunk.blocks.0.mlp.fc1.weight", "backbone.vision_backbone.trunk.blocks.0.mlp.fc1.bias", "backbone.vision_backbone.trunk.blocks.0.mlp.fc2.weight", "backbone.vision_backbone.trunk.blocks.0.mlp.fc2.bias", "backbone.vision_backbone.trunk.blocks.1.norm1.weight", "backbone.vision_backbone.trunk.blocks.1.norm1.bias", "backbo